# Study 942 — The Inverse Tax — the teardown

The replicate's accounting, the HAC *t* and block-bootstrap CI on the daily gap, the financing-passthrough regression, the break-even rebate credit, the era and rate-regime cuts, both proxy sweeps, the path-drag table, and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `b9485964fb9d`).

## The two arms, stated exactly

Per $1 of NAV, no timing signal anywhere:

1. **fund** — the inverse ETF's own daily total return (SH, PSQ, SDS).
2. **direct** — a directly-short book rebalanced daily to the same −k× NAV:

$$ r^{direct}_t \;=\; k\,r^{index,TR}_t \;+\; c\,|k|\,rf_t \;-\; \text{borrow}_t \;-\; \text{cost}_t $$

The **total-return** leg is deliberate: a share-short owes the distributions. `rf` is the `^IRX` act/360 accrual over calendar days at the *previous* close's level. **One execution lag, applied identically to both arms:** the exposure set at the close of `t−1` earns day `t`'s return and finances at `t−1`'s rate.

`c` (rebate credit) and the borrow fee are **PROXY / ASSUMPTION** inputs, not tape — both are swept below, and the break-even `c` is reported.

> 💡 **In plain words:** we rebuild, day by day, the P&L of someone who shorted the index themselves — paying the dividends, paying to borrow the shares, earning whatever interest their broker hands back — and put it next to the ETF.

In [1]:
R = {'start': '2007-05-31', 'end': '2026-06-30', 'n_days': 4799, 'fp': 'b9485964fb9d', 'credit': 1.0, 'borrow_bps': 30.0, 'cost_bps': 1.0, 'irx_ann': 1.47, 'bil_ann': 1.36, 'irx_bil_diff': 0.11, 'sh_gap': 1.06, 'sh_t': 5.1, 'sh_t_iid': 2.1, 'sh_sd': 13.8, 'sh_ci_lo': 0.71, 'sh_ci_hi': 1.45, 'psq_gap': 0.58, 'psq_t': 2.5, 'psq_sd': 16.6, 'psq_ci_lo': 0.2, 'psq_ci_hi': 0.97, 'sds_gap': 1.19, 'sds_t': 3.78, 'sds_sd': 21.6, 'sds_ci_lo': 0.69, 'sds_ci_hi': 1.74, 'sh_sm': -0.79, 'sh_sm_t': -2.76, 'psq_sm': -0.23, 'psq_sm_t': -0.94, 'sds_sm': -2.51, 'sds_sm_t': -4.87, 'be_sm_sh': 0.46, 'be_sm_psq': 0.84, 'be_sm_sds': 0.14, 'sm_zirp': -2.52, 'sm_zirp_t': -6.61, 'sm_norm': 1.59, 'sm_norm_t': 4.2, 'hac_profile': ((0, 2.1), (1, 2.82), (2, 3.21), (5, 4.61), (9, 5.1), (21, 5.5), (63, 4.34), (126, 3.27), (252, 2.44)), 'gap_acf1': -0.44, 'sh_exsharpe': -0.574, 'dir_exsharpe': -0.627, 'sharpe_diff': 0.052, 'sh_cagr': -11.22, 'dir_cagr': -12.16, 'sh_vol': 19.7, 'dir_vol': 19.8, 'sh_dd': -94.7, 'dir_dd': -95.4, 'sh_raw': 2.18, 'sh_div': 1.85, 'sh_fin': 1.74, 'sh_er': 0.89, 'sh_resid': -0.52, 'sh_beta': 0.9914, 'sh_gamma': 1.187, 'sh_gamma_px': 1.4, 'sh_alpha': 0.33, 'psq_raw': 1.69, 'psq_div': 0.81, 'psq_fin': 2.01, 'psq_er': 0.95, 'psq_resid': -0.18, 'psq_beta': 0.9917, 'psq_gamma': 1.371, 'psq_gamma_px': 1.263, 'sds_raw': 3.39, 'sds_div': 3.7, 'sds_fin': 2.8, 'sds_er': 0.89, 'sds_resid': -2.21, 'sds_beta': 0.9823, 'sds_gamma': 0.954, 'sds_gamma_px': 1.164, 'spy_yield': 1.85, 'qqq_yield': 0.81, 'be_sh': 1.72, 'be_psq': 1.39, 'be_sds': 1.41, 'era_e_n': 2163, 'era_e_gap': 0.16, 'era_e_t': 0.39, 'era_e_rf': 0.49, 'era_l_n': 2636, 'era_l_gap': 1.79, 'era_l_t': 9.54, 'era_l_rf': 2.26, 'zirp_n': 2785, 'zirp_gap': -0.46, 'zirp_t': -1.83, 'zirp_rf': 0.15, 'norm_n': 2014, 'norm_gap': 3.15, 'norm_t': 11.37, 'norm_rf': 3.29, 'c0_b0': 2.22, 'c0_b0_t': 8.74, 'c1_b30': 1.06, 'c1_b30_t': 5.1, 'c15_b0': 0.02, 'c15_b0_t': 0.09, 'c2_b0': -0.72, 'c2_b0_t': -3.81, 'c2_b30': -0.41, 'c2_b30_t': -2.19, 'cost0': 1.01, 'cost0_t': 4.9, 'cost10': 1.42, 'cost10_t': 6.82, 'sh_drag_63': -0.34, 'sh_drag_63_med': -0.26, 'sh_wrap_63': 0.27, 'sds_drag_63': -1.12, 'sds_drag_63_med': -0.76, 'sds_wrap_63': 0.38, 'sds_drag_252': -2.68, 'sh_drag_252': -0.8, 'syn_plant': -2.77, 'syn_plant_t': -22.06, 'syn_null': 0.23, 'syn_null_t': 1.84, 'syn_null_mean': 0.027, 'syn_null_sd': 0.159, 'syn_null_fire': 0, 'syn_plant_mean': -2.973, 'syn_seeds': 12}

## Headline — the daily gap

In [2]:
for tag, g, t, sd, lo, hi in [
    ('SH  (-1x SPY)', R['sh_gap'],  R['sh_t'],  R['sh_sd'],  R['sh_ci_lo'],  R['sh_ci_hi']),
    ('PSQ (-1x QQQ)', R['psq_gap'], R['psq_t'], R['psq_sd'], R['psq_ci_lo'], R['psq_ci_hi']),
    ('SDS (-2x SPY)', R['sds_gap'], R['sds_t'], R['sds_sd'], R['sds_ci_lo'], R['sds_ci_hi']),
]:
    print(f"{tag}: gap {g:+.2f}%/yr  HAC t {t:+.2f}  daily sd {sd:.1f} bps  "
          f"boot 95% CI [{lo:+.2f}, {hi:+.2f}]")
print(f"\nn={R['n_days']:,} days, {R['start']} -> {R['end']}, "
      f"c={R['credit']:.0f}, borrow={R['borrow_bps']:.0f} bps, cost={R['cost_bps']:.0f} bp")
print(f"excess-of-cash (minus BIL): fund exSharpe {R['sh_exsharpe']:+.3f} vs "
      f"direct {R['dir_exsharpe']:+.3f} (diff {R['sharpe_diff']:+.3f}); "
      f"absolute CAGR {R['sh_cagr']:+.2f}% vs {R['dir_cagr']:+.2f}%")
print(f"financing cross-check: ^IRX accrual {R['irx_ann']:.2f}%/yr vs BIL TR "
      f"{R['bil_ann']:.2f}%/yr (wedge {R['irx_bil_diff']:+.2f} pp)")

SH  (-1x SPY): gap +1.06%/yr  HAC t +5.10  daily sd 13.8 bps  boot 95% CI [+0.71, +1.45]
PSQ (-1x QQQ): gap +0.58%/yr  HAC t +2.50  daily sd 16.6 bps  boot 95% CI [+0.20, +0.97]
SDS (-2x SPY): gap +1.19%/yr  HAC t +3.78  daily sd 21.6 bps  boot 95% CI [+0.69, +1.74]

n=4,799 days, 2007-05-31 -> 2026-06-30, c=1, borrow=30 bps, cost=1 bp
excess-of-cash (minus BIL): fund exSharpe -0.574 vs direct -0.627 (diff +0.052); absolute CAGR -11.22% vs -12.16%
financing cross-check: ^IRX accrual 1.47%/yr vs BIL TR 1.36%/yr (wedge +0.11 pp)


Both arms are short books over a sample in which the index rose roughly five-fold, so both excess Sharpes are deeply negative and neither is interpretable on its own. The **gap** is the estimand.

## The same-mandate counterweight — where the sign comes from

The headline race is *product vs product*. It is **not** an exposure-matched race: SH/PSQ/SDS track the **price** index, while a share-short is short the **total-return** stream and owes the distributions. That term is the largest single piece of the raw gap. Debit the fund `|k| ×` the realised distribution yield and both arms sit on the total-return leg — the strictly like-for-like comparison:

In [3]:
hdr = f"{'fund':5s} {'headline':>10s} {'same-mandate':>14s} {'HAC t':>8s} {'c* (same)':>10s}"
print(hdr); print('-'*len(hdr))
for f, h, sm, t, c in [
    ('SH',  R['sh_gap'],  R['sh_sm'],  R['sh_sm_t'],  R['be_sm_sh']),
    ('PSQ', R['psq_gap'], R['psq_sm'], R['psq_sm_t'], R['be_sm_psq']),
    ('SDS', R['sds_gap'], R['sds_sm'], R['sds_sm_t'], R['be_sm_sds']),
]:
    print(f"{f:5s} {h:+10.2f} {sm:+14.2f} {t:+8.2f} {c:10.2f}")
print(f"\nSH same-mandate by regime: ZIRP {R['sm_zirp']:+.2f}%/yr "
      f"(t {R['sm_zirp_t']:+.2f})   bills>=1% {R['sm_norm']:+.2f}%/yr "
      f"(t {R['sm_norm_t']:+.2f})")

fund    headline   same-mandate    HAC t  c* (same)
---------------------------------------------------
SH         +1.06          -0.79    -2.76       0.46
PSQ        +0.58          -0.23    -0.94       0.84
SDS        +1.19          -2.51    -4.87       0.14

SH same-mandate by regime: ZIRP -2.52%/yr (t -6.61)   bills>=1% +1.59%/yr (t +4.20)


**The estimand's sign is a definitional choice and both signs clear |*t*| ≥ 2.** Neither framing is wrong; quoting only one would be. The stamp is Mixed for exactly this reason, and no robustness cut rescues a single sign.

## How much of the *t* is the HAC bandwidth?

The daily gap's lag-1 autocorrelation is **-0.44** — the ETF's closing print against the index close, bid-ask bounce and premium/discount noise that reverses the next day. HAC therefore *reduces* the variance relative to i.i.d., so the automatic-bandwidth *t* is not comparable to a naive one. The profile, not the point:

In [4]:
print('Bartlett lags -> HAC t on the SH gap (0 = i.i.d.)')
for lag, t in R['hac_profile']:
    mark = '   <- automatic bandwidth' if lag == 9 else ''
    print(f"  {lag:4d}: {t:+.2f}{mark}")
ts = [t for _, t in R['hac_profile']]
print(f"\nrange {min(ts):+.2f} to {max(ts):+.2f}; "
      f"|t|>=2 at every truncation, but the conservative read is the i.i.d. one.")

Bartlett lags -> HAC t on the SH gap (0 = i.i.d.)
     0: +2.10
     1: +2.82
     2: +3.21
     5: +4.61
     9: +5.10   <- automatic bandwidth
    21: +5.50
    63: +4.34
   126: +3.27
   252: +2.44

range +2.10 to +5.50; |t|>=2 at every truncation, but the conservative read is the i.i.d. one.


## Decomposition of the raw gap (c = 0, no borrow, no cost)

γ is the OLS loading of the fund's daily return on `|k|·rf` in

$$ r^{fund}_t = \alpha + \beta\,(k\,r^{index,TR}_t) + \gamma\,(|k|\,rf_t) + \varepsilon_t $$

so the financing passthrough is **measured**, not assumed. The expense ratio is the one stated (non-tape) term; the residual absorbs swap spread, reset slippage and tracking error.

In [5]:
hdr = f"{'fund':5s} {'raw':>7s} {'divs':>7s} {'financing':>10s} {'-ER':>7s} {'residual':>9s} {'beta':>7s} {'gamma':>7s}"
print(hdr); print('-'*len(hdr))
for f, raw, dv, fin, er, res, b, g in [
    ('SH',  R['sh_raw'],  R['sh_div'],  R['sh_fin'],  R['sh_er'],  R['sh_resid'],  R['sh_beta'],  R['sh_gamma']),
    ('PSQ', R['psq_raw'], R['psq_div'], R['psq_fin'], R['psq_er'], R['psq_resid'], R['psq_beta'], R['psq_gamma']),
    ('SDS', R['sds_raw'], R['sds_div'], R['sds_fin'], R['sds_er'], R['sds_resid'], R['sds_beta'], R['sds_gamma']),
]:
    print(f"{f:5s} {raw:+7.2f} {dv:+7.2f} {fin:+10.2f} {-er:+7.2f} {res:+9.2f} {b:7.4f} {g:7.3f}")
print(f"\nall %/yr. index yields: SPY {R['spy_yield']:.2f}%, QQQ {R['qqq_yield']:.2f}%; "
      f"bills {R['irx_ann']:.2f}%")
print('\nSPEC CHECK - gamma re-estimated on the PRICE leg (what the funds track):')
for f, kabs, g_tr, g_px in [('SH', 1.0, R['sh_gamma'], R['sh_gamma_px']),
                            ('PSQ', 1.0, R['psq_gamma'], R['psq_gamma_px']),
                            ('SDS', 2.0, R['sds_gamma'], R['sds_gamma_px'])]:
    print(f"  {f:4s} gamma {g_tr:.3f} (TR leg) vs {g_px:.3f} (price leg) "
          f"-> financing term moves {(g_px-g_tr)*kabs*R['irx_ann']:+.2f}%/yr")
print('  the RAW GAP is spec-free; only the financing/residual SPLIT moves.')

fund      raw    divs  financing     -ER  residual    beta   gamma
------------------------------------------------------------------
SH      +2.18   +1.85      +1.74   -0.89     -0.52  0.9914   1.187
PSQ     +1.69   +0.81      +2.01   -0.95     -0.18  0.9917   1.371
SDS     +3.39   +3.70      +2.80   -0.89     -2.21  0.9823   0.954

all %/yr. index yields: SPY 1.85%, QQQ 0.81%; bills 1.47%

SPEC CHECK - gamma re-estimated on the PRICE leg (what the funds track):
  SH   gamma 1.187 (TR leg) vs 1.400 (price leg) -> financing term moves +0.31%/yr
  PSQ  gamma 1.371 (TR leg) vs 1.263 (price leg) -> financing term moves -0.16%/yr
  SDS  gamma 0.954 (TR leg) vs 1.164 (price leg) -> financing term moves +0.62%/yr
  the RAW GAP is spec-free; only the financing/residual SPLIT moves.


β ≈ 0.99 on the −1× funds and 0.9823 on the −2×: the funds deliver essentially their stated exposure, so the gap is *not* tracking failure. γ = **1.187** (SH) and **1.371** (PSQ) — between one and two units of the bill rate on NAV, which is the crux: a retail direct shorter is at `c = 1`.

> 💡 **In plain words:** the funds hand back more short-rate interest than an ordinary shorter is credited, and they never pay the index's dividends. Those two credits are worth more than the fees they charge.

## Break-even rebate credit

The gap is exactly linear in `c` (each unit costs the direct book `|k|·rf`), so the indifference point is a single number the reader can place their own account against.

In [6]:
for f, c in [('SH', R['be_sh']), ('PSQ', R['be_psq']), ('SDS', R['be_sds'])]:
    print(f"{f:4s}: c* = {c:.2f} units of the bill rate")
print('\nretail account ~ c=1.0  ->  the FUND wins')
print('prime brokerage ~ c=2.0 ->  the DIRECT SHORT wins')

SH  : c* = 1.72 units of the bill rate
PSQ : c* = 1.39 units of the bill rate
SDS : c* = 1.41 units of the bill rate

retail account ~ c=1.0  ->  the FUND wins
prime brokerage ~ c=2.0 ->  the DIRECT SHORT wins


## Robustness — era cut and rate-regime cut

The rate-regime partition uses the *previous* close's 13-week bill level, known at the start of each day: an ex-ante split, not a look-ahead sort.

In [7]:
print(f"2007-2015 (n={R['era_e_n']:,}, bills {R['era_e_rf']:.2f}%/yr): "
      f"gap {R['era_e_gap']:+.2f}%/yr  t {R['era_e_t']:+.2f}")
print(f"2016-2026 (n={R['era_l_n']:,}, bills {R['era_l_rf']:.2f}%/yr): "
      f"gap {R['era_l_gap']:+.2f}%/yr  t {R['era_l_t']:+.2f}")
print()
print(f"bills <1%  (n={R['zirp_n']:,}, avg {R['zirp_rf']:.2f}%/yr): "
      f"gap {R['zirp_gap']:+.2f}%/yr  t {R['zirp_t']:+.2f}  <- the folklore's regime")
print(f"bills >=1% (n={R['norm_n']:,}, avg {R['norm_rf']:.2f}%/yr): "
      f"gap {R['norm_gap']:+.2f}%/yr  t {R['norm_t']:+.2f}")

2007-2015 (n=2,163, bills 0.49%/yr): gap +0.16%/yr  t +0.39
2016-2026 (n=2,636, bills 2.26%/yr): gap +1.79%/yr  t +9.54

bills <1%  (n=2,785, avg 0.15%/yr): gap -0.46%/yr  t -1.83  <- the folklore's regime
bills >=1% (n=2,014, avg 3.29%/yr): gap +3.15%/yr  t +11.37


The gap is not a constant — it is a rate story, and the era cut is the same statement in calendar clothing. Note that even in ZIRP, its own best regime, the claimed tax reaches only *t* = -1.83: it is never *robustly* demonstrated anywhere on this tape.

## Proxy sweeps — the honest sensitivity

In [8]:
print('SH, gap %/yr (HAC t) across the two PROXY inputs:')
print(f"  c=0.0, borrow=  0 bps : {R['c0_b0']:+.2f} ({R['c0_b0_t']:+.2f})")
print(f"  c=1.0, borrow= 30 bps : {R['c1_b30']:+.2f} ({R['c1_b30_t']:+.2f})   <- base case")
print(f"  c=1.5, borrow=  0 bps : {R['c15_b0']:+.2f} ({R['c15_b0_t']:+.2f})   <- indifference")
print(f"  c=2.0, borrow= 30 bps : {R['c2_b30']:+.2f} ({R['c2_b30_t']:+.2f})")
print(f"  c=2.0, borrow=  0 bps : {R['c2_b0']:+.2f} ({R['c2_b0_t']:+.2f})   <- the ONLY corner")
print( "                                             that reproduces the claimed tax")
print()
print('rebalance-cost sweep on the direct book (c=1, 30 bps borrow):')
print(f"  gross (0 bps): {R['cost0']:+.2f}%/yr (t {R['cost0_t']:+.2f})   "
      f"10 bps: {R['cost10']:+.2f}%/yr (t {R['cost10_t']:+.2f})")
print('  costs only WIDEN the fund advantage - it is the direct book that must be re-levered daily')

SH, gap %/yr (HAC t) across the two PROXY inputs:
  c=0.0, borrow=  0 bps : +2.22 (+8.74)
  c=1.0, borrow= 30 bps : +1.06 (+5.10)   <- base case
  c=1.5, borrow=  0 bps : +0.02 (+0.09)   <- indifference
  c=2.0, borrow= 30 bps : -0.41 (-2.19)
  c=2.0, borrow=  0 bps : -0.72 (-3.81)   <- the ONLY corner
                                             that reproduces the claimed tax

rebalance-cost sweep on the direct book (c=1, 30 bps borrow):
  gross (0 bps): +1.01%/yr (t +4.90)   10 bps: +1.42%/yr (t +6.82)
  costs only WIDEN the fund advantage - it is the direct book that must be re-levered daily


## Path drag — and its correct owner

Non-overlapping holding windows. `daily-reset − static` is the pure constant-leverage path drag of Cheng-Madhavan / Avellaneda-Zhang; `fund − daily-reset` is what the wrapper itself adds.

In [9]:
print(f"SH  63d: daily-reset - static {R['sh_drag_63']:+.2f} pp "
      f"(med {R['sh_drag_63_med']:+.2f})  |  fund - daily-reset {R['sh_wrap_63']:+.2f} pp")
print(f"SDS 63d: daily-reset - static {R['sds_drag_63']:+.2f} pp "
      f"(med {R['sds_drag_63_med']:+.2f})  |  fund - daily-reset {R['sds_wrap_63']:+.2f} pp")
print(f"SH 252d: {R['sh_drag_252']:+.2f} pp     SDS 252d: {R['sds_drag_252']:+.2f} pp")
print()
print('ratio of the -2x drag to the -1x drag at 63d: %.1fx  (theory (k^2-k)/2 -> 3x)'
      % (R['sds_drag_63'] / R['sh_drag_63']))

SH  63d: daily-reset - static -0.34 pp (med -0.26)  |  fund - daily-reset +0.27 pp
SDS 63d: daily-reset - static -1.12 pp (med -0.76)  |  fund - daily-reset +0.38 pp
SH 252d: -0.80 pp     SDS 252d: -2.68 pp

ratio of the -2x drag to the -1x drag at 63d: 3.3x  (theory (k^2-k)/2 -> 3x)


The drag scales with |k| about as `(k² − k)/2 · σ²` predicts. But it sits entirely in the **`daily-reset − static`** column — the column a self-managed constant-leverage book pays too. The wrapper's own column is small and slightly positive.

> 💡 **In plain words:** the decay is real, and it is the price of keeping your short the same size every day. Buying the fund does not create it; doing it yourself does not avoid it.

## Live synthetic control — the machinery is unbiased

Offline, no cache, fixed seeds. A simulated fund with a **planted** 3%/yr tax must be caught with the right sign and size; a costless replicate must read zero. This proves the harness measures what it claims — it never supports the real-tape stamp.

In [10]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from inverse_tax import data, strategy as st
seeds = tuple(942 + s for s in range(12))
null  = [st.synthetic_detect(p) for p, _ in data.synthetic_panel(seeds=seeds, signal_strength=0.0)]
plant = [st.synthetic_detect(p) for p, _ in data.synthetic_panel(seeds=seeds, signal_strength=1.0)]
ng = np.array([d['gap_ann_pct'] for d in null]);  nt = np.array([d['gap_t'] for d in null])
pg = np.array([d['gap_ann_pct'] for d in plant])
print('null  x%d: gap mean %+.3f%%/yr (sd %.3f), |t|>=2 on %d/%d seeds'
      % (len(seeds), ng.mean(), ng.std(ddof=1), (abs(nt) >= 2).sum(), len(seeds)))
print('plant x%d: gap mean %+.3f%%/yr (sd %.3f) against a planted -3.00, right sign %d/%d'
      % (len(seeds), pg.mean(), pg.std(ddof=1), (pg < 0).sum(), len(seeds)))

null  x12: gap mean +0.027%/yr (sd 0.159), |t|>=2 on 0/12 seeds
plant x12: gap mean -2.973%/yr (sd 0.159) against a planted -3.00, right sign 12/12


In [11]:
d = st.decompose(st.synthetic_frame(
        data.synthetic_daily(signal_strength=1.0, tax_ann_full=0.01,
                             div_yield_ann=0.02, passthrough=1.5,
                             rate_ann=0.04, seed=942)[0]),
        fund='FUND', index='IDX', k=-1.0, expense_ratio_pct=0.0)
print('decomposition on a tape with a planted 2.00% dividend yield and a 1.500 passthrough:')
print('  recovered dividend yield %.2f%%   recovered gamma %.3f   beta %.4f'
      % (d['div_yield_ann_pct'], d['gamma'], d['beta']))

decomposition on a tape with a planted 2.00% dividend yield and a 1.500 passthrough:
  recovered dividend yield 2.00%   recovered gamma 1.518   beta 0.9997


## What this does not establish

- **Survivorship.** SH, PSQ and SDS survived from 2006-2007 to today; the industry has liquidated a long tail of inverse and leveraged-inverse products whose wrapper residuals are not in this sample. The −0.52 %/yr residual on SH is a *survivor's* residual, and the three tickers are an explicit, hindsight-flavoured universe pick (largest and longest-lived of their kind).
- **No recall / buy-in risk** on the direct book: it is assumed able to borrow SPY and QQQ every day at a flat fee. That charge is zero here and biases *against* the fund, whose shareholder cannot be recalled.
- **No tax, no margin mechanics**, and the bill rate is `^IRX`, a discount quote rather than a broker rate sheet (BIL cross-check bounds the error at 0.11 pp/yr).
- **The financing/residual split is spec-dependent** (γ on the TR leg vs the price leg); the raw gap is not.

## Verdict

- **Signal — Mixed.** The structural gap is real and precisely estimated: **+1.06 %/yr** on SH with HAC *t* = **+5.10** (i.i.d. *t* +2.10) and a bootstrap CI [+0.71, +1.45] clear of zero, replicated on PSQ (+0.58, *t* +2.50) and SDS (+1.19, *t* +3.78). But **its sign is a choice, not a measurement**: on the same-mandate race it is -0.79 %/yr (*t* -2.76) on SH and -2.51 (*t* -4.87) on SDS — the folklore's sign, also significant. On top of that the break-even rebate credit is **1.72** units of the bill rate, the gap swings from -0.46 %/yr in ZIRP to +3.15 %/yr when bills pay 3.3%, and the early era is a coin-flip (+0.16 %/yr, *t* +0.39). Mixed, not Real.
- **Tradability — Fragile.** Bankable only as an implementation choice on a hedge sleeve: both arms lost 11.2%/yr and 12.2%/yr absolute over a sample in which the index quintupled. The plausible range of the two proxies spans the sign change, the edge evaporates at zero rates, and the −2× fund still carries a -2.21 %/yr wrapper residual on top of -1.12 pp a quarter of reset drag.
- **The claim, itemised.** Expense ratio: real, and the *smallest* term. Daily-reset drag: real, and not the wrapper's — a self-managed constant-leverage book pays it identically. Financing-and-dividends: the leg that carries the whole claim, and on this tape it is genuinely two-sided — it favours the fund at retail terms on a product-vs-product comparison, and favours the direct short for a prime broker, at zero rates, or on a same-mandate comparison.